# - Qlora
# ---- Quantization + LoRA
# --- instead of full precision values for the matrix it rounds it up as a result of that memory usage is saved heavily . Because to stor efull precision data we need 16 bit weights...however with quantization we can only store 4 bit weights


# - Lora
# ---- We do a partial training ...instead of training the entire model architecture we train only small part of the transformer / LLM matrices / layers . Much faster and much cost effective

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

# HR Domain Fine tuned CHatbot

In [ ]:
# Dummy training data
data = [
    {
        "text": "### Human: What is the maternity leave policy?\n ### Assistant : Employees get 24 weeks of paid maternity leave"
    },
    {
        "text": "### Human: How many casual leaves am I eligible for ?\n #### Assistant: You receive 20 casual leaves annually"
    },
    {
        "text": "### Human: What is the notice period am I supposed to serve? \n### Assistant: You have to serve standard notice period of 90 days "
    },
    {
        "text": "### Human: Can I work from home ? \n### Assistant: Work from home is only allowed based on manager approval in exceptional cases "
    },
    {
        "text": "### Human: What is the reimbursement policy around internet and telecom bills ? \n### Assistant: we pay you salary and you take care of your internet and telecom bill . No reimbursement available. "
    }


]

dataset = Dataset.from_list(data)
dataset

Dataset({
    features: ['text'],
    num_rows: 5
})

In [ ]:
# Load a base model for finetuning
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
# QloRA First
# We will load the above model in 4 bit as a result of that memory will be saved

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4_bit = True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# loRA Config
peft_config = LoraConfig(
    r = 4, # Rank of the low rank matrices that will changed by loRA... Weight vectors
    lora_alpha = 32, #scaling factor of lora updates
    target_modules = ["q_proj","v_proj"], # Which layers inside transformer is trained --- Query projection , Value Projection ( other layers = key and output projections)
    lora_dropout = 0.05, #prevents overfitting
    bias = "none", # no bias term
    task_type = "CAUSAL_LM" #next token generation model
)

training_args = TrainingArguments(
    output_dir = "./results",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,
    learning_rate = 2e-4,
    num_train_epochs = 3,
    logging_steps = 1,
    save_strategy = "no",
    fp16 = True,
    report_to = "none"
)

trainer = SFTTrainer(
    model = model, # Quantized model
    train_dataset = dataset,
    peft_config = peft_config, # We are telling which layers to train
    args = training_args
)


Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,3.351993


In [ ]:
trainer.model.save_pretrained("hr-chatbot")
tokenizer.save_pretrained("hr-chatbot")

In [ ]:
# Inferencing
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model = "hr-chatbot",
    tokenizer = tokenizer,
    max_new_tokens = 100
)


prompt = "### Human : What is the maternity leave policy?\n### Assistant:"
print(pipe(prompt)[0]["generated_text"])